# Characterize nodules at their ground-truth locations with MedGemma

Runs `5b_characterize_ground_truth_nodules.py`: for every pylidc consensus nodule already recorded in `ground_truth_annotations.json` (patients 1-20), crops directly around that nodule's own real centroid/diameter (no detector involved) and asks MedGemma 1.5 to rate it on the same 9 pylidc attributes as `5_characterize_nodules.py` - then compares straight against the mean of that nodule's own radiologist annotations. No MONAI/simpleitk needed, since there's no detector step.

Before running anything:
1. **Runtime > Change runtime type > GPU** (T4 is fine).
2. Upload `lidc_idri_p1-20.zip` to your Google Drive (same zip used for the counting notebook - patients 1-20 + `annotations.csv`, ~1.1GB). Skip this if you already have it there from before.
3. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) holding a Hugging Face access token that has accepted the MedGemma license at https://huggingface.co/google/medgemma-1.5-4b-it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/freya-gul/rail.git
%cd rail

Point this at wherever you uploaded `lidc_idri_p1-20.zip` in Drive. It unzips into `datasets/LDIC-IDRI-subset/` inside the cloned repo — that's the path this script's `DICOM_ROOT` already expects:

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/lidc_idri_p1-20.zip"  # <-- update to your actual upload path
DATA_DIR = "datasets/LDIC-IDRI-subset"  # relative to the repo root (we've already %cd'd into rail)

import pathlib
assert pathlib.Path(ZIP_PATH).exists(), f"{ZIP_PATH} not found — check the path/upload"

In [ ]:
!mkdir -p {DATA_DIR}
!unzip -q {ZIP_PATH} -d {DATA_DIR}
!ls {DATA_DIR}

In [ ]:
# No monai/simpleitk needed - this script never touches the MONAI detector, only
# crops directly out of the raw DICOM series at the ground-truth nodule locations.
!pip install -q pydicom "transformers>=5.12.1" "huggingface_hub>=1.21.0"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

Sanity check: GPU visible to torch (the script already defaults to `cuda` > `mps` > `cpu`):

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Run characterization

Prints live progress per nodule (predicted vs. ground-truth attribute values aren't shown inline, but pass/fail on JSON validation is) plus a running ETA. Resumable at the individual-nodule level — a killed run picks back up partway through a patient rather than redoing it.

In [ ]:
!python image_download/5b_characterize_ground_truth_nodules.py --start 1 --end 20

## Results

- `image_download/nodule_characteristics_gt/<patient>.json` — per-nodule predicted attributes, ground-truth mean, error, raw MedGemma response, and JSON-validation problems (if any).
- `image_download/characterize_ground_truth_comparison.csv` — the same data flattened across all patients, one row per nodule.
- `image_download/characterize_ground_truth_summary.json` — MAE and signed bias per attribute, aggregated across every nodule.

Quick look at the summary table:

In [ ]:
import json
summary = json.load(open("image_download/characterize_ground_truth_summary.json"))
for attr, stats in summary.items():
    print(f"{attr:<18} MAE={stats['mae']:.2f}  bias={stats['bias']:+.2f}  n={stats['n']}")